# 🏦 PDF Data Extraction — PRIIPS KID — Solutions

> ⚠️ This file contains the **solutions**. Try first with `exercise_pdf_lists.ipynb`!

## 0. Setup — run this first

We only import the tools. **No PDF is generated**: we read the real documents from `assets/`.
The path is written relative to this notebook (`../../assets`).

In [ ]:
import os
import re
import pandas as pd
import fitz  # PyMuPDF

ASSETS_DIR = "../../assets"   # the assets/ folder at the repo root
print("Files available in assets/:", os.listdir(ASSETS_DIR))

---
## Exercise 1 — Load only the PRIIPS file(s)

In [ ]:
all_files = os.listdir(ASSETS_DIR)
priips_files = [f for f in all_files if f.lower().endswith('.pdf') and 'priips' in f.lower()]
print("PRIIPS files:", priips_files)
print("Count:", len(priips_files))

---
## Exercise 2 — Read the PDF text

In [ ]:
path = os.path.join(ASSETS_DIR, priips_files[0])

doc = fitz.open(path)
print("Number of pages:", doc.page_count)
full_text = "".join(page.get_text() for page in doc)
doc.close()

print(full_text[:500])

---
## Exercise 3 — Extract the identification fields

In [ ]:
isin      = re.search(r'\b([A-Z]{2}[A-Z0-9]{9}\d)\b', full_text).group(1)
currency  = re.search(r'Currency:\s*([A-Z]{3})', full_text).group(1)
published = re.search(r'published on (\d{2}/\d{2}/\d{4})', full_text).group(1)

print("ISIN     :", isin)
print("Currency :", currency)
print("Published:", published)

---
## Exercise 4 — Extract the risk & horizon fields

In [ ]:
sri           = int(re.search(r'classified this product as (\d)\s+out of 7', full_text).group(1))
holding_years = int(re.search(r'Recommended holding period\s*:?\s*(\d+)\s*years?', full_text).group(1))

level = "low" if sri <= 2 else "medium" if sri <= 4 else "high"
print(f"SRI: {sri}/7  ({level} risk)")
print("Recommended holding period:", holding_years, "years")

---
## Exercise 5 — Build a structured summary

In [ ]:
kid = {
    "file": priips_files[0],
    "isin": isin,
    "currency": currency,
    "published": published,
    "sri": sri,
    "holding_years": holding_years,
}
pd.DataFrame([kid])

---
## Exercise 6 — Generalise with a function

In [ ]:
def extract_kid(path):
    """Open a PRIIPS KID PDF and return its key fields as a dict."""
    doc = fitz.open(path)
    text = "".join(page.get_text() for page in doc)
    doc.close()

    def grab(pattern, cast=str):
        m = re.search(pattern, text)
        return cast(m.group(1)) if m else None

    return {
        "file": os.path.basename(path),
        "isin": grab(r'\b([A-Z]{2}[A-Z0-9]{9}\d)\b'),
        "currency": grab(r'Currency:\s*([A-Z]{3})'),
        "published": grab(r'published on (\d{2}/\d{2}/\d{4})'),
        "sri": grab(r'classified this product as (\d)\s+out of 7', int),
        "holding_years": grab(r'Recommended holding period\s*:?\s*(\d+)\s*years?', int),
    }

rows = [extract_kid(os.path.join(ASSETS_DIR, f)) for f in priips_files]
df_kids = pd.DataFrame(rows)
df_kids.to_excel("kid_summary.xlsx", index=False)
print("Exported kid_summary.xlsx with", len(df_kids), "row(s).")
df_kids